<a href="https://colab.research.google.com/github/TambCoys/Laboratory-of-Data-Science/blob/main/split%2B_perfetto_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import json
import csv
import xml.etree.ElementTree as ET
import os
from datetime import datetime # Import datetime for date parsing

# -------------------------------
#  CONFIG
# -------------------------------
JSON_FILE = "tracks_cleaned.json"
XML_FILE = "artists_cleaned.xml"
OUTPUT_DIR = "warehouse_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------------
#  COUNTERS (For Surrogate Keys)
# -------------------------------
cnt_geo = 1
cnt_artist = 1
cnt_album = 1
cnt_date = 1
cnt_symphony = 1
cnt_text = 1
cnt_feature = 1
cnt_fact = 1

# -------------------------------
#  LOOKUP DICTIONARIES
# -------------------------------
map_geo = {}        # (country, region...) -> int
map_artist = {}     # 'ART123' -> int
map_album = {}      # 'ALB123' -> int
map_date = {}       # '20210405' -> int
map_symphony = {}   # 'TRK123' -> int (1:1 with Track)
map_text = {}       # 'TRK123' -> int (1:1 with Track)
map_feature = {}    # 'TRK123_F0' -> int

# -------------------------------
#  DATA BUFFERS
# -------------------------------
rows_geo = []
rows_artist = []
rows_album = []
rows_date = []
rows_symphony = []
rows_text = []
rows_features = []
rows_fact = []

# -------------------------------
#  HELPERS
# -------------------------------
def format_date_components(y, m, d):
    if y != None and m != None and d != None:
        Y = f"{int(y):04d}"
        M = f"{int(m):02d}"
        D = f"{int(y):04d}{int(m):02d}{int(d):02d}"
        return Y, M, D
    return None, None, None

def determine_season(month):
    if month != None:
        m = int(month)
        if m in [12,1,2]: return "Winter"
        if m in [3,4,5]: return "Spring"
        if m in [6,7,8]: return "Summer"
        return "Autumn"
    return None

def write_csv(filename, header, rows):
    path = os.path.join(OUTPUT_DIR, filename)
    print(f"Writing {len(rows)} rows to {filename}...")
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)

def get_or_create_date_fk(year, month, day):
    global cnt_date, map_date, rows_date

    Y_str, M_str, D_str = format_date_components(year, month, day)

    # Handle None dates by defaulting to
    if D_str is None:
        D_str = "NULL"
        Y_str = "NULL"
        M_str = "NULL"
        month_int = 0 # For season determination
    else:
        month_int = int(month) if month is not None else None

    if D_str not in map_date:
        map_date[D_str] = cnt_date
        rows_date.append([cnt_date, D_str, Y_str, M_str, determine_season(month_int)])
        cnt_date += 1

    return map_date[D_str]

# ==========================================
#  PHASE 1: LOAD SOURCE FILES
# ==========================================
print("Loading Source Files...")
with open(JSON_FILE, "r", encoding="utf-8") as f:
    data_tracks = json.load(f)

tree = ET.parse(XML_FILE)
data_artists_xml = tree.getroot()

# ==========================================
#  PHASE 2: BUILD DIMENSIONS
# ==========================================

# --- 2A. PROCESS XML (Artists & Geography) ---
print("Processing XML Dimensions...")
for row in data_artists_xml.findall(".//row"):
    # 1. Geography
    country = row.findtext("country")
    region = row.findtext("region")
    province = row.findtext("province")
    city = row.findtext("province") # Using province as city fallback
    h3 = row.findtext("h3_idx")

    geo_key = (country, region, province, city, h3)

    if geo_key not in map_geo:
        map_geo[geo_key] = cnt_geo
        rows_geo.append([cnt_geo, h3, country, region, province, city])
        cnt_geo += 1

    # 2. Artist
    artist_natural_id = row.findtext("id_author")
    if artist_natural_id not in map_artist:
        map_artist[artist_natural_id] = cnt_artist

        birth_date_str = row.findtext("birth_date")
        birth_year, birth_month, birth_day = None, None, None

        if birth_date_str:
            try:
                # Attempt to parse 'YYYY-MM-DD'
                parsed_date = datetime.strptime(birth_date_str, '%Y-%m-%d')
                birth_year = parsed_date.year
                birth_month = parsed_date.month
                birth_day = parsed_date.day
            except ValueError:
                # Handle other potential formats or invalid dates if necessary
                # For now, if parsing fails, year, month, day remain None
                pass

        birth_date_fk = get_or_create_date_fk(birth_year, birth_month, birth_day)

        rows_artist.append([
            cnt_artist,                 # Surrogate PK
            map_geo[geo_key],           # Geo FK (Surrogate)
            row.findtext("name"),
            row.findtext("gender"),
            birth_date_fk,              # Store FK instead of raw string
            row.findtext("birth_place"),
            row.findtext("nationality")
        ])
        cnt_artist += 1

# --- 2B. PROCESS JSON (Album, Date, Symphony, Text, Features) ---
print("Processing JSON Dimensions...")
for t in data_tracks:
    track_natural_id = t["id"]

    # 1. Album
    alb_nat_id = t["id_album"]
    if alb_nat_id and alb_nat_id not in map_album:
        map_album[alb_nat_id] = cnt_album
        rows_album.append([
            cnt_album,                  # Surrogate PK
            t["album_name"],
            t["album_type"],
            t["album_release_date"]
        ])
        cnt_album += 1

    # 2. Date
    get_or_create_date_fk(t["year"], t["month"], t["day"])

    # 3. Symphony
    if track_natural_id not in map_symphony:
        map_symphony[track_natural_id] = cnt_symphony
        rows_symphony.append([
            cnt_symphony,               # Surrogate PK
            t["bpm"], t["rolloff"], t["flux"], t["rms"], t["flatness"],
            t["spectral_complexity"], t["pitch"], t["loudness"]
        ])
        cnt_symphony += 1

    # 4. Text
    if track_natural_id not in map_text:
        map_text[track_natural_id] = cnt_text
        rows_text.append([
            cnt_text,                   # Surrogate PK
            t["n_sentences"], t["n_tokens"], t["char_per_tok"],
            t["avg_token_per_clause"], 1 if t["explicit"] else 0
        ])
        cnt_text += 1

    # 5. Features
    # Logic: Generate Surrogate for every feature entry.
    # Note: We track the LAST generated feature ID for this track to link to Fact.
    last_feature_pk_for_track = None

    if t["featured_artists"]:
        feats = [x.strip() for x in t["featured_artists"].split(",")]
        for i, fartist in enumerate(feats):
            feat_nat_key = f"{track_natural_id}_F{i}"

            if feat_nat_key not in map_feature:
                map_feature[feat_nat_key] = cnt_feature
                rows_features.append([cnt_feature, track_natural_id, fartist])
                cnt_feature += 1

            last_feature_pk_for_track = map_feature[feat_nat_key]
    else:
        feat_nat_key = f"{track_natural_id}_F0"
        if feat_nat_key not in map_feature:
            map_feature[feat_nat_key] = cnt_feature
            rows_features.append([cnt_feature, track_natural_id, None])
            cnt_feature += 1
        last_feature_pk_for_track = map_feature[feat_nat_key]

# ==========================================
#  PHASE 3: BUILD FACT TABLE
# ==========================================
print("Building Fact Table...")
# We loop through JSON again (or could have buffered) to build facts
# using the maps we fully populated above.

for t in data_tracks:
    track_natural_id = t["id"]

    # RESOLVE KEYS
    # Artist
    art_nat_id = t["id_artist"]
    art_fk = map_artist.get(art_nat_id)

    # Album
    alb_nat_id = t["id_album"]
    alb_fk = map_album.get(alb_nat_id)

    # Date
    y, m, d = t["year"], t["month"], t["day"]
    _, _, D = format_date_components(y, m, d)
    if D is None: D = "NULL"
    date_fk = map_date.get(D)

    # Symphony & Text (1:1)
    sym_fk = map_symphony.get(track_natural_id)
    txt_fk = map_text.get(track_natural_id)

    # Feature
    # Re-derive the key for the LAST feature to match Phase 2B logic
    if t["featured_artists"]:
        count = len([x for x in t["featured_artists"].split(",")])
        feat_key = f"{track_natural_id}_F{count-1}"
    else:
        feat_key = f"{track_natural_id}_F0"
    feat_fk = map_feature.get(feat_key)

    # Only create fact if main links exist (Artist & Album)
    if art_fk and alb_fk:
        rows_fact.append([
            #track_natural_id,
            t["title"],
            t["language"],
            t["duration_ms"],
            t["streams@1month"],
            t["popularity"],
            None, # Category
            art_fk,                 # Surrogate FK
            alb_fk,                 # Surrogate FK
            date_fk,                # Surrogate FK
            sym_fk,                 # Surrogate FK
            txt_fk,                 # Surrogate FK
            feat_fk                 # Surrogate FK
        ])

# ==========================================
#  PHASE 4: WRITE OUTPUTS
# ==========================================
print("Writing CSVs...")

write_csv("ArtistGeoDim.csv",
    ["ArtistGeoCodePK", "H3_index", "Country", "Region", "Province", "City"],
    rows_geo)

write_csv("ArtistDim.csv",
    ["ArtistCodePK", "ArtistGeoCodeFK", "Name", "Gender", "ArtistBirthDateFK", "BirthPlace", "Nationality"], # Updated header
    rows_artist)

write_csv("AlbumDim.csv",
    ["AlbumCodePK", "Title", "Type", "ReleaseDate"],
    rows_album)

write_csv("DateDim.csv",
    ["DateCodePK", "DateInt", "Year", "Month", "Season"],
    rows_date)

write_csv("SymphonyDim.csv",
    ["SymphonyCodePK", "BPM", "Rolloff", "Flux", "RMS", "Flatness", "Spectral_Complexity", "Pitch", "Loudness"],
    rows_symphony)

write_csv("TextDim.csv",
    ["TextCodePK", "N_Sentences", "N_Tokens", "Char_Per_Tok", "Avg_Token_Per_Clause", "Is_Explicit"],
    rows_text)

write_csv("SongsFeatures.csv",
    ["FeatureCodePK", "FeaturedArtistName"],
    rows_features)

write_csv("Published_Song_fact.csv",
    ["Title", "Language", "Duration", "Streams_1month", "Popularity",
     "ArtistCodeFK", "AlbumCodeFK", "DateCodeFK", "SymphonyCodeFK", "TextCodeFK", "FeatureCodeFK"],
    rows_fact)

print("DONE! All tables now use Integer Surrogate Keys.")

Loading Source Files...
Processing XML Dimensions...
Processing JSON Dimensions...
Building Fact Table...
Writing CSVs...
Writing 31 rows to ArtistGeoDim.csv...
Writing 104 rows to ArtistDim.csv...
Writing 3061 rows to AlbumDim.csv...
Writing 2500 rows to DateDim.csv...
Writing 11093 rows to SymphonyDim.csv...
Writing 11093 rows to TextDim.csv...
Writing 12097 rows to SongsFeatures.csv...
Writing 11088 rows to Published_Song_fact.csv...
DONE! All tables now use Integer Surrogate Keys.
